In [14]:
import pandas as pd
import numpy as np
import datetime
import json
import os
from IPython.display import display_html
from IPython.display import HTML
import csv
import dataframe_image as dfi
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [27]:
def display_horizontal(*args):
    html_str = ''
    for df in args:
        html_str += df.to_html()
    display_html(
        html_str.replace('table','table style="display:inline;margin-right:50px"'), 
        raw=True
    )

def export_horizontal_matplotlib(df1, df2,
                                  title1="A股多投", 
                                  title2="", 
                                  title3="3年",
                                  filename=None):
    df1 = df1.round(4)
    df2 = df2.round(4)

    # Ensure output directory exists
    output_dir = os.path.join(".", "指数清单", "最新分表")
    os.makedirs(output_dir, exist_ok=True)

    # Generate filename if not provided
    if filename is None:
        filename = f"{title1}-{title3}.png"
    filepath = os.path.join(output_dir, filename)

    # Stack data
    left_data = np.vstack([df1.columns.values.reshape(1, -1), df1.values.astype(str)])
    right_data = np.vstack([df2.columns.values.reshape(1, -1), df2.values.astype(str)])
    n_rows = max(left_data.shape[0], right_data.shape[0])

    # Pad shorter table
    if left_data.shape[0] < n_rows:
        left_data = np.vstack([
            left_data,
            np.full((n_rows - left_data.shape[0], left_data.shape[1]), "", dtype=object)
        ])
    if right_data.shape[0] < n_rows:
        right_data = np.vstack([
            right_data,
            np.full((n_rows - right_data.shape[0], right_data.shape[1]), "", dtype=object)
        ])

    # Create spacer
    spacer = np.full((n_rows, 1), "", dtype=object)
    if title1:
        spacer[0, 0] = title1
    if title2:
        spacer[1, 0] = title2
    if title3:
        spacer[2, 0] = title3

    full_table = np.hstack([left_data, spacer, right_data])

    fig, ax = plt.subplots(figsize=(16, 0.5 * n_rows + 1.5))
    ax.axis('off')
    table = ax.table(cellText=full_table,
                     loc='center',
                     cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)

    col_widths = [0.10, 0.16, 0.08, 0.10, 0.10, 0.16, 0.08]
    for (row, col), cell in table.get_celld().items():
        if col < len(col_widths):
            cell.set_width(col_widths[col])
        cell.set_linewidth(0.3)
        if row == 0 and col != 3:
            cell.set_fontsize(13)
            cell.set_text_props(weight='bold')
            cell.set_height(cell.get_height() * 1.4)

    fig.canvas.draw()
    bbox = table.get_window_extent(fig.canvas.get_renderer()).expanded(1.02, 1.05)
    fig.savefig(
        filepath,
        dpi=300,
        bbox_inches=bbox.transformed(fig.dpi_scale_trans.inverted())
    )
    plt.close()

In [28]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)
    
with open("INDEX_NAME.json", 'r', encoding = 'utf-8') as f:
    INDEX_NAME = json.load(f)

In [29]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [30]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)

In [31]:
latest_date_str = INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')

with open("指数清单/最新分表/latest_date.txt", "w", encoding="utf-8") as f:
    f.write(latest_date_str)

In [32]:
UNIVERSE_A = {}
UNIVERSE_O = {}
target_dates = {}

for ticker in INDEX_DATA:
    if ticker in a_share_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_A[ticker] = INDEX_DATA[ticker]

    elif ticker in other_market_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_O[ticker] = INDEX_DATA[ticker]

In [33]:
print(f"Assets under Analysis: {len(UNIVERSE_A)}  {len(UNIVERSE_O)}")
print(f"Total Assets: {len(INDEX_DATA)}")

Assets under Analysis: 73  35
Total Assets: 186


In [34]:
n = 0
for ticker in other_market_dict:
    if ticker in INDEX_DATA:
        n += 1

n

39

<br><br><br><br>
## 多投列表

In [35]:
UNIVERSE = UNIVERSE_A

for years in [1, 3]:
    Z_PE_TTM = {}
    Z_DIVIDENDYIELD = {}
    duration = int(years * 252)

    for ticker in UNIVERSE:
        target_date = target_dates[ticker]
        if len(UNIVERSE[ticker]) < duration:
            continue

        # PE_TTM Z-score
        sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
        Z_PE_TTM[ticker] = Z

        # DIVIDENDYIELD2 Z-score
        sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
        Z_DIVIDENDYIELD[ticker] = Z

    display_number = 15

    df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
    df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

    df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=True)
    df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=False)

    top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
    top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

    top_Z_PE_TTM["指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
    top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

    top_Z_DIVIDENDYIELD["指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
    top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

    export_horizontal_matplotlib(
        top_Z_PE_TTM,
        top_Z_DIVIDENDYIELD,
        title1="A股多投",
        title2=INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d'),
        title3=f"{years}年"
    )

In [36]:
UNIVERSE = UNIVERSE_O

for years in [1, 3]:
    Z_PE_TTM = {}
    Z_DIVIDENDYIELD = {}
    duration = int(years * 252)

    for ticker in UNIVERSE:
        target_date = target_dates[ticker]
        if len(UNIVERSE[ticker]) < duration:
            continue

        # PE_TTM Z-score
        sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
        sample_data.dropna(inplace=True)
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
        Z_PE_TTM[ticker] = Z

        # DIVIDENDYIELD2 Z-score
        sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
        sample_data.dropna(inplace=True)
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
        Z_DIVIDENDYIELD[ticker] = Z

    display_number = 15

    df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
    df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

    df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=True)
    df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=False)

    top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
    top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

    top_Z_PE_TTM["指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
    top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

    top_Z_DIVIDENDYIELD["指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
    top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

    export_horizontal_matplotlib(
        top_Z_PE_TTM,
        top_Z_DIVIDENDYIELD,
        title1="海外多投",
        title2=INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d'),
        title3=f"{years}年"
    )

C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_25172\3957363398.py:21: RuntimeWarning: invalid value encountered in scalar divide
  Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_25172\3957363398.py:32: RuntimeWarning: invalid value encountered in scalar divide
  Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std


<br><br><br><br>
## 空投列表

In [37]:
UNIVERSE = UNIVERSE_A

for years in [1, 3]:
    Z_PE_TTM = {}
    Z_DIVIDENDYIELD = {}
    duration = int(years * 252)

    for ticker in UNIVERSE:
        target_date = target_dates[ticker]
        if len(UNIVERSE[ticker]) < duration:
            continue

        # PE TTM 分位值
        sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
        Z_PE_TTM[ticker] = Z

        # 股息率 分位值
        sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
        Z_DIVIDENDYIELD[ticker] = Z

    display_number = 15

    df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
    df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

    # Changed sort order for 空投
    df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=False)
    df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=True)

    top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
    top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

    top_Z_PE_TTM["指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
    top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

    top_Z_DIVIDENDYIELD["指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
    top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

    export_horizontal_matplotlib(
        top_Z_PE_TTM,
        top_Z_DIVIDENDYIELD,
        title1="A股空投",
        title2=INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d'),
        title3=f"{years}年"
    )


In [38]:
UNIVERSE = UNIVERSE_O

for years in [1, 3]:
    Z_PE_TTM = {}
    Z_DIVIDENDYIELD = {}
    duration = int(years * 252)

    for ticker in UNIVERSE:
        target_date = target_dates[ticker]
        if len(UNIVERSE[ticker]) < duration:
            continue

        # PE_TTM 分位值
        sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
        sample_data.dropna(inplace=True)
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
        Z_PE_TTM[ticker] = Z

        # 股息率 分位值
        sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
        sample_data.dropna(inplace=True)
        if len(sample_data) != duration:
            continue

        mean = sample_data.mean()
        std = sample_data.std()
        Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
        Z_DIVIDENDYIELD[ticker] = Z

    display_number = 15

    df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
    df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

    # Sort for 空投逻辑
    df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=False)
    df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=True)

    top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
    top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

    top_Z_PE_TTM["指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
    top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

    top_Z_DIVIDENDYIELD["指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
    top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

    export_horizontal_matplotlib(
        top_Z_PE_TTM,
        top_Z_DIVIDENDYIELD,
        title1="海外空投",
        title2=INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d'),
        title3=f"{years}年"
    )


C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_25172\2938195467.py:21: RuntimeWarning: invalid value encountered in scalar divide
  Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_25172\2938195467.py:32: RuntimeWarning: invalid value encountered in scalar divide
  Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
